# WCCI Episodic Survival Plots

This notebook mirrors the A0 episodic-survival notebook, but it reads only the permanent WCCI `outputs/run_data` folders: `wcci_aib_24h`, `wcci_sparse16`, `wcci_hvg`, `wcci_baseline`, and `wcci_aib`. Edit the run-group cell to change which curves are compared.

In [2]:

from pathlib import Path
import importlib
import json
import re
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


def find_task_dir(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "main.py").is_file() and candidate.name == "Topology_Task":
            return candidate
        task_dir = candidate / "Topology_Task"
        if (task_dir / "main.py").is_file():
            return task_dir
    raise RuntimeError("Could not locate Topology_Task/main.py")


TASK_DIR = find_task_dir()
HELPER_DIR = TASK_DIR / "analysis" / "metrics" / "helpers"
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

from run_data import scan_run_data
import wandb_metrics as wm
wm = importlib.reload(wm)

RUN_DATA_ROOT = TASK_DIR / "outputs" / "run_data"
FIG_DIR = TASK_DIR / "outputs" / "wcci_metric_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

WCCI_GROUPS = [
    "wcci_aib_2",
    "wcci_sparse16_2",
    "wcci_hvg_2",
    "wcci_baseline_2",
]
SEEDS = (0, 1, 2)
AGENTS = ["agent_0", "agent_1", "agent_2", "agent_3"]
SAVE_FIGURES = True
SHOW_FIGURES = True

# Reuse the same editable plotting helpers as baseline_episodic_survival_plots.ipynb,
# but keep all data loading local from outputs/run_data.
wm.configure_task_dir(TASK_DIR)
wm.FIG_DIR = FIG_DIR
wm.SHOW_FIGURES = SHOW_FIGURES

print("Task dir:", TASK_DIR)
print("Run data root:", RUN_DATA_ROOT)
print("Figure dir:", FIG_DIR)
print("WCCI groups:", WCCI_GROUPS)
print("wandb_metrics:", wm.__file__)


Task dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task
Run data root: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data
WCCI groups: ['wcci_aib_2', 'wcci_sparse16_2', 'wcci_hvg_2', 'wcci_baseline_2']


## Load Downloaded WCCI Histories

In [3]:
def _path_or_none(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    path = Path(value).expanduser()
    return path if path.exists() else None


def read_history(row):
    parquet_path = _path_or_none(row.get("history_parquet"))
    csv_path = _path_or_none(row.get("history_csv"))
    if parquet_path is not None:
        history = pd.read_parquet(parquet_path)
    elif csv_path is not None:
        history = pd.read_csv(csv_path)
    else:
        raise FileNotFoundError(f"No local history file for {row.get('run_name')}")
    history = history.copy()
    history["run_name"] = row["run_name"]
    history["run_id"] = row["run_id"]
    history["download_group"] = row["download_group"]
    history["run_state"] = row.get("state")
    return history


run_index_all = scan_run_data(RUN_DATA_ROOT, include_legacy=False)
run_index = run_index_all[run_index_all["download_group"].isin(WCCI_GROUPS)].copy()
if run_index.empty:
    raise RuntimeError(f"No WCCI runs found under {RUN_DATA_ROOT}. Check that the downloaded groups exist.")

histories = []
for row in run_index.to_dict("records"):
    try:
        histories.append(read_history(row))
    except Exception as exc:
        print(f"Skipped {row.get('run_name')}: {exc}")

history_df = pd.concat(histories, ignore_index=True, sort=False) if histories else pd.DataFrame()
if history_df.empty:
    raise RuntimeError("No WCCI histories could be loaded.")

coverage = (
    run_index.assign(
        family=lambda df: df["run_name"].map(lambda name: re.sub(r"_s\\d+$", "", str(name))),
        seed=lambda df: df["run_name"].map(lambda name: int(re.search(r"_s(\\d+)$", str(name)).group(1)) if re.search(r"_s(\\d+)$", str(name)) else np.nan),
    )
    .groupby(["download_group", "family", "state"], dropna=False, as_index=False)
    .agg(
        runs=("run_name", "count"),
        seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
        rows_min=("rows", "min"),
        rows_max=("rows", "max"),
    )
    .sort_values(["download_group", "family", "state"])
)
print(f"Loaded {len(run_index)} runs and {len(history_df):,} history rows.")
display(coverage)


Loaded 30 runs and 43,153 history rows.


,download_group,family,state,runs,seeds,rows_min,rows_max
0,wcci_aib_2,wcci_aib_00_flat_local_t020_72x576_lr20m_f010_s0,finished,1,[],1446,1446
1,wcci_aib_2,wcci_aib_00_flat_local_t020_72x576_lr20m_f010_s1,finished,1,[],1446,1446
2,wcci_aib_2,wcci_aib_00_flat_local_t020_72x576_lr20m_f010_s2,finished,1,[],1446,1446
3,wcci_aib_2,wcci_aib_01_flat_local_t010_72x576_lr20m_f010_s0,crashed,1,[],1355,1355
4,wcci_aib_2,wcci_aib_01_flat_local_t010_72x576_lr20m_f010_s1,crashed,1,[],1373,1373
5,wcci_aib_2,wcci_aib_01_flat_local_t010_72x576_lr20m_f010_s2,crashed,1,[],1383,1383
6,wcci_baseline_2,wcci_mlp_baseline_512x512x512_72x576_lr20m_f01...,finished,1,[],1446,1446
7,wcci_baseline_2,wcci_mlp_baseline_512x512x512_72x576_lr20m_f01...,finished,1,[],1446,1446
8,wcci_baseline_2,wcci_mlp_baseline_512x512x512_72x576_lr20m_f01...,finished,1,[],1446,1446
9,wcci_baseline_2,wcci_mlp_baseline_72x576_lr20m_f010_s0,finished,1,[],1446,1446


## Labels and Run Helpers

In [4]:

def safe_name(text):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(text)).strip("._-") or "plot"


def save_figure(fig, name):
    if fig is None or not SAVE_FIGURES:
        return None
    path = FIG_DIR / f"{safe_name(name)}.html"
    fig.write_html(path, include_plotlyjs="cdn")
    print("Saved:", path)
    return path


def seed_from_run(run_name):
    match = re.search(r"_s(\d+)$", str(run_name))
    return int(match.group(1)) if match else np.nan


def family_from_run(run_name):
    return re.sub(r"_s\d+$", "", str(run_name))


WCCI_LABELS = {
    # Baselines from wcci_baseline_2
    "wcci_mlp_baseline_72x576": "MLP baseline 72x576",
    "wcci_mlp_baseline_72x576_lr20m_f010": "MLP baseline 72x576 lr20M f0.10",
    "wcci_mlp_baseline_512x512x512_72x576_lr20m_f010": "MLP baseline 512x3 lr20M f0.10",
    # Heuristic / gate from wcci_hvg_2
    "wcci_hvg_01_eval_rho090_72x576_lr20m_f010": "global rho heuristic 0.90",
    "wcci_hvg_04_eval_local_rho090_72x576_lr20m_f010": "local rho heuristic 0.90",
    "wcci_hvg_02_gate_final_map_72x576_lr20m_f010": "gate final-action MAP",
    "wcci_hvg_03_gate_hierarchical_72x576_lr20m_f010": "gate hierarchical greedy",
    # Sparse penalty from wcci_sparse16_2
    "wcci_sparse16_flat_p010_72x576_lr20m_f010": "Sparse16 flat p0.010",
    # Adaptive intervention budget from wcci_aib_2
    "wcci_aib_00_flat_local_t020_72x576_lr20m_f010": "AIB local budget t0.20",
    "wcci_aib_01_flat_local_t010_72x576_lr20m_f010": "AIB local budget t0.10",
}

WCCI_COLOR = {
    "MLP baseline 72x576": "#4e79a7",
    "MLP baseline 72x576 lr20M f0.10": "#1f77b4",
    "MLP baseline 512x3 lr20M f0.10": "#17becf",
    "global rho heuristic 0.90": "#ff7f0e",
    "local rho heuristic 0.90": "#2ca02c",
    "gate final-action MAP": "#9467bd",
    "gate hierarchical greedy": "#d62728",
    "Sparse16 flat p0.010": "#8c564b",
    "AIB local budget t0.20": "#bcbd22",
    "AIB local budget t0.10": "#e377c2",
}


def pretty_label(family):
    return WCCI_LABELS.get(str(family), str(family).replace("wcci_", "").replace("_", " "))


def seeded(prefix, seeds=SEEDS):
    return [f"{prefix}_s{seed}" for seed in seeds]


def available_run_names(history=None):
    data = history_df if history is None else history
    return set(data["run_name"].dropna().astype(str).unique())


def report_missing_runs(run_groups, history=None):
    available = available_run_names(history)
    rows = []
    for label, runs in run_groups.items():
        missing = [run for run in runs if run not in available]
        rows.append({"curve": label, "expected": len(runs), "available": len(runs) - len(missing), "missing": missing})
    coverage = pd.DataFrame(rows)
    display(coverage)
    return coverage


def infer_family_table():
    rows = []
    for run_name in sorted(history_df["run_name"].dropna().astype(str).unique()):
        rows.append({
            "download_group": history_df.loc[history_df["run_name"].eq(run_name), "download_group"].iloc[0],
            "run_name": run_name,
            "family": family_from_run(run_name),
            "label": pretty_label(family_from_run(run_name)),
            "seed": seed_from_run(run_name),
        })
    return pd.DataFrame(rows)


family_table = infer_family_table()
display(family_table.sort_values(["download_group", "family", "seed"]))


,download_group,run_name,family,label,seed
0,wcci_aib_2,wcci_aib_00_flat_local_t020_72x576_lr20m_f010_s0,wcci_aib_00_flat_local_t020_72x576_lr20m_f010,AIB flat local t0.20 lr20m f0.10,0
1,wcci_aib_2,wcci_aib_00_flat_local_t020_72x576_lr20m_f010_s1,wcci_aib_00_flat_local_t020_72x576_lr20m_f010,AIB flat local t0.20 lr20m f0.10,1
2,wcci_aib_2,wcci_aib_00_flat_local_t020_72x576_lr20m_f010_s2,wcci_aib_00_flat_local_t020_72x576_lr20m_f010,AIB flat local t0.20 lr20m f0.10,2
3,wcci_aib_2,wcci_aib_01_flat_local_t010_72x576_lr20m_f010_s0,wcci_aib_01_flat_local_t010_72x576_lr20m_f010,aib 01 flat local t010 72x576 lr20m f010,0
4,wcci_aib_2,wcci_aib_01_flat_local_t010_72x576_lr20m_f010_s1,wcci_aib_01_flat_local_t010_72x576_lr20m_f010,aib 01 flat local t010 72x576 lr20m f010,1
5,wcci_aib_2,wcci_aib_01_flat_local_t010_72x576_lr20m_f010_s2,wcci_aib_01_flat_local_t010_72x576_lr20m_f010,aib 01 flat local t010 72x576 lr20m f010,2
18,wcci_baseline_2,wcci_mlp_baseline_512x512x512_72x576_lr20m_f01...,wcci_mlp_baseline_512x512x512_72x576_lr20m_f010,mlp baseline 512x512x512 72x576 lr20m f010,0
19,wcci_baseline_2,wcci_mlp_baseline_512x512x512_72x576_lr20m_f01...,wcci_mlp_baseline_512x512x512_72x576_lr20m_f010,mlp baseline 512x512x512 72x576 lr20m f010,1
20,wcci_baseline_2,wcci_mlp_baseline_512x512x512_72x576_lr20m_f01...,wcci_mlp_baseline_512x512x512_72x576_lr20m_f010,mlp baseline 512x512x512 72x576 lr20m f010,2
24,wcci_baseline_2,wcci_mlp_baseline_72x576_s0,wcci_mlp_baseline_72x576,mlp baseline 72x576,0


## Survival Helpers

In [5]:
SURVIVAL_METRICS = {
    "test": ["test/episodic_survival", "test/charts/episodic_survival"],
    "train_eval": ["train_eval/episodic_survival", "train_eval/charts/episodic_survival"],
    "legacy": ["charts/episodic_survival"],
}


def first_existing_metric(candidates, data=None):
    data = history_df if data is None else data
    for metric in candidates:
        if metric in data.columns:
            return metric
    raise KeyError(f"None of these metrics are available: {candidates}")


def _survival_scale(values):
    numeric = pd.to_numeric(values, errors="coerce")
    max_value = numeric.max(skipna=True)
    if pd.isna(max_value):
        return 100.0
    return 100.0 if max_value <= 1.5 else 1.0


def survival_long(run_groups, *, split="test", smooth=5, history=None):
    data = history_df if history is None else history
    metric = first_existing_metric(SURVIVAL_METRICS[split], data)
    rows = []
    for label, runs in run_groups.items():
        for run_name in runs:
            run_data = data[data["run_name"].astype(str).eq(str(run_name))]
            if run_data.empty:
                continue
            cols = ["run_name", "run_id", "download_group", "_step", metric]
            cols = [col for col in cols if col in run_data.columns]
            curve = run_data[cols].copy().dropna(subset=[metric, "_step"])
            if curve.empty:
                continue
            scale = _survival_scale(curve[metric])
            curve["value"] = pd.to_numeric(curve[metric], errors="coerce") * scale
            curve["step_millions"] = pd.to_numeric(curve["_step"], errors="coerce") / 1_000_000.0
            curve["curve"] = label
            curve["seed"] = seed_from_run(run_name)
            curve["family"] = family_from_run(run_name)
            curve["metric"] = metric
            curve = curve.sort_values("_step")
            if smooth and smooth > 1:
                curve["value_smooth"] = curve["value"].rolling(int(smooth), min_periods=1).mean()
            else:
                curve["value_smooth"] = curve["value"]
            rows.append(curve)
    return pd.concat(rows, ignore_index=True, sort=False) if rows else pd.DataFrame()


def mean_survival_curves(long_df):
    if long_df.empty:
        return pd.DataFrame()
    return (
        long_df
        .groupby(["curve", "_step", "step_millions"], as_index=False, observed=True)
        .agg(
            mean_survival=("value_smooth", "mean"),
            std_survival=("value_smooth", "std"),
            min_survival=("value_smooth", "min"),
            max_survival=("value_smooth", "max"),
            n_seeds=("seed", "nunique"),
            seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
        )
    )


def plot_survival_groups(run_groups, *, title, split="test", smooth=5, y_range=(0, 105), show_members=True, show_std=True, save_name=None, history=None):
    long_df = survival_long(run_groups, split=split, smooth=smooth, history=history)
    if long_df.empty:
        print(f"No survival data for {title}")
        return None, long_df, pd.DataFrame()
    mean_df = mean_survival_curves(long_df)
    fig = go.Figure()
    for label, runs in run_groups.items():
        color = WCCI_COLOR.get(label, None)
        curve_data = mean_df[mean_df["curve"].eq(label)].sort_values("step_millions")
        if curve_data.empty:
            continue
        if show_members:
            for run_name, member in long_df[long_df["curve"].eq(label)].groupby("run_name", sort=False):
                member = member.sort_values("step_millions")
                fig.add_trace(go.Scatter(
                    x=member["step_millions"],
                    y=member["value_smooth"],
                    mode="lines",
                    name=f"{label} seed {seed_from_run(run_name)}",
                    legendgroup=label,
                    showlegend=False,
                    line={"color": color, "width": 1.2},
                    opacity=0.18,
                    hovertemplate=(
                        f"<b>{label}</b><br>run={run_name}<br>"
                        "step=%{x:.2f}M<br>survival=%{y:.2f}%<extra></extra>"
                    ),
                ))
        if show_std and curve_data["n_seeds"].max() > 1:
            std = curve_data["std_survival"].fillna(0.0)
            fig.add_trace(go.Scatter(
                x=pd.concat([curve_data["step_millions"], curve_data["step_millions"].iloc[::-1]]),
                y=pd.concat([curve_data["mean_survival"] + std, (curve_data["mean_survival"] - std).iloc[::-1]]),
                fill="toself",
                fillcolor=color if color else "rgba(31,119,180,0.14)",
                line={"color": "rgba(255,255,255,0)"},
                opacity=0.12,
                hoverinfo="skip",
                showlegend=False,
                legendgroup=label,
            ))
        fig.add_trace(go.Scatter(
            x=curve_data["step_millions"],
            y=curve_data["mean_survival"],
            mode="lines",
            name=label,
            legendgroup=label,
            line={"color": color, "width": 3.5},
            customdata=np.stack([curve_data["n_seeds"], curve_data["seeds"].astype(str)], axis=-1),
            hovertemplate=(
                f"<b>{label}</b><br>step=%{{x:.2f}}M<br>"
                "mean survival=%{y:.2f}%<br>n_seeds=%{customdata[0]}<br>seeds=%{customdata[1]}<extra></extra>"
            ),
        ))
    fig.update_layout(
        title=title,
        template="plotly_white",
        width=1350,
        height=650,
        xaxis_title="steps (M)",
        yaxis_title=f"{split} episodic survival (%)",
        hovermode="x unified",
        legend=dict(
            x=0.985,
            y=0.035,
            xanchor="right",
            yanchor="bottom",
            bgcolor="rgba(255,255,255,0.88)",
            bordercolor="rgba(80,80,80,0.25)",
            borderwidth=1,
            font=dict(size=14),
        ),
        margin={"l": 80, "r": 40, "t": 90, "b": 70},
    )
    if y_range is not None:
        fig.update_yaxes(range=list(y_range))
    save_figure(fig, save_name or title)
    if SHOW_FIGURES:
        fig.show()
    return fig, long_df, mean_df


def latest_survival_summary(run_groups, *, split="test", last_n=5, history=None):
    long_df = survival_long(run_groups, split=split, smooth=1, history=history)
    if long_df.empty:
        return pd.DataFrame()
    per_run = (
        long_df.sort_values(["run_name", "_step"])
        .groupby(["curve", "run_name", "seed"], as_index=False, observed=True)
        .tail(int(last_n))
        .groupby(["curve", "run_name", "seed"], as_index=False, observed=True)
        .agg(final_survival=("value", "mean"), final_step_m=("step_millions", "max"))
    )
    summary = (
        per_run.groupby("curve", as_index=False, observed=True)
        .agg(
            mean_survival=("final_survival", "mean"),
            std_survival=("final_survival", "std"),
            min_survival=("final_survival", "min"),
            max_survival=("final_survival", "max"),
            n_seeds=("seed", "nunique"),
            seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
            max_step_m=("final_step_m", "max"),
        )
        .sort_values("mean_survival", ascending=False)
    )
    return summary


## Editable WCCI Run Groups

In [6]:

# Edit this cell to choose which WCCI runs appear in each figure.
# This matches the style of baseline_episodic_survival_plots.ipynb:
# - use prefix/name/regex/runs/run_dir to select runs
# - use label/color/dash/width to control how the curve is displayed
# - use WCCI_COMPARISON_SPECS to decide which curves appear in each plot
# The notebook only uses downloaded permanent run_data, no W&B API calls.

WCCI_RUN_SPECS = {
    # Baselines from wcci_baseline_2
    "baseline_72x576": {
        "prefix": "wcci_mlp_baseline_72x576_s",
        "label": "MLP baseline 72x576",
        "color": "#4e79a7",
        "width": 3.2,
    },
    "baseline_72x576_lr20m_f010": {
        "prefix": "wcci_mlp_baseline_72x576_lr20m_f010_s",
        "label": "MLP baseline 72x576 lr20M f0.10",
        "color": "#1f77b4",
        "width": 4.0,
    },
    "baseline_512x3_lr20m_f010": {
        "prefix": "wcci_mlp_baseline_512x512x512_72x576_lr20m_f010_s",
        "label": "MLP baseline 512x3 lr20M f0.10",
        "color": "#17becf",
        "width": 3.2,
    },

    # Heuristic / gate from wcci_hvg_2
    "hvg_global_rho090": {
        "prefix": "wcci_hvg_01_eval_rho090_72x576_lr20m_f010_s",
        "label": "global rho heuristic 0.90",
        "color": "#ff7f0e",
    },
    "hvg_local_rho090": {
        "prefix": "wcci_hvg_04_eval_local_rho090_72x576_lr20m_f010_s",
        "label": "local rho heuristic 0.90",
        "color": "#2ca02c",
    },
    "gate_final_map": {
        "prefix": "wcci_hvg_02_gate_final_map_72x576_lr20m_f010_s",
        "label": "gate final-action MAP",
        "color": "#9467bd",
    },
    "gate_hierarchical": {
        "prefix": "wcci_hvg_03_gate_hierarchical_72x576_lr20m_f010_s",
        "label": "gate hierarchical greedy",
        "color": "#d62728",
    },

    # Sparse action cost from wcci_sparse16_2
    "sparse16_flat_p010": {
        "prefix": "wcci_sparse16_flat_p010_72x576_lr20m_f010_s",
        "label": "Sparse16 flat p0.010",
        "color": "#8c564b",
    },

    # Adaptive intervention budget from wcci_aib_2
    "aib_local_t020": {
        "prefix": "wcci_aib_00_flat_local_t020_72x576_lr20m_f010_s",
        "label": "AIB local budget t0.20",
        "color": "#bcbd22",
    },
    "aib_local_t010": {
        "prefix": "wcci_aib_01_flat_local_t010_72x576_lr20m_f010_s",
        "label": "AIB local budget t0.10",
        "color": "#e377c2",
    },
}

BASELINE_SPEC_KEY = "baseline_72x576_lr20m_f010"

WCCI_COMPARISON_SPECS = {
    "WCCI baselines": {
        "sources": ["wcci_baseline_2"],
        "run_keys": [
            "baseline_72x576",
            "baseline_72x576_lr20m_f010",
            "baseline_512x3_lr20m_f010",
        ],
        "title": "WCCI baselines: test episodic survival",
        "subplot_title": "baseline variants",
        "save_name": "wcci_2_baselines_test_survival_editable",
        "show_members": True,
        "show_std": True,
    },
    "WCCI HVG": {
        "sources": ["wcci_baseline_2", "wcci_hvg_2"],
        "run_keys": [
            BASELINE_SPEC_KEY,
            "hvg_global_rho090",
            "hvg_local_rho090",
            "gate_final_map",
            "gate_hierarchical",
        ],
        "title": "WCCI baseline vs heuristic/gate: test episodic survival",
        "subplot_title": "baseline vs heuristic/gate",
        "save_name": "wcci_2_baseline_vs_hvg_test_survival_editable",
        "show_members": True,
        "show_std": True,
    },
    "WCCI Sparse16": {
        "sources": ["wcci_baseline_2", "wcci_sparse16_2"],
        "run_keys": [BASELINE_SPEC_KEY, "sparse16_flat_p010"],
        "title": "WCCI baseline vs Sparse16: test episodic survival",
        "subplot_title": "baseline vs Sparse16",
        "save_name": "wcci_2_baseline_vs_sparse16_test_survival_editable",
        "show_members": True,
        "show_std": True,
    },
    "WCCI AIB": {
        "sources": ["wcci_baseline_2", "wcci_aib_2"],
        "run_keys": [BASELINE_SPEC_KEY, "aib_local_t020", "aib_local_t010"],
        "title": "WCCI baseline vs AIB: test episodic survival",
        "subplot_title": "baseline vs AIB",
        "save_name": "wcci_2_baseline_vs_aib_test_survival_editable",
        "show_members": True,
        "show_std": True,
    },
    "WCCI all controls": {
        "sources": ["wcci_baseline_2", "wcci_hvg_2", "wcci_sparse16_2", "wcci_aib_2"],
        "run_keys": [
            BASELINE_SPEC_KEY,
            "hvg_global_rho090",
            "hvg_local_rho090",
            "gate_final_map",
            "gate_hierarchical",
            "sparse16_flat_p010",
            "aib_local_t020",
            "aib_local_t010",
        ],
        "title": "WCCI baseline vs all control variants: test episodic survival",
        "subplot_title": "baseline vs all controls",
        "save_name": "wcci_2_baseline_vs_all_controls_test_survival_editable",
        "show_members": False,
        "show_std": True,
    },
}


def comparison_sources(comparison_name):
    return WCCI_COMPARISON_SPECS.get(comparison_name, {}).get("sources", WCCI_GROUPS)


def source_history(comparison_name):
    return history_df[history_df["download_group"].isin(comparison_sources(comparison_name))].copy()


def specs_for_keys(keys):
    return [dict(WCCI_RUN_SPECS[key]) for key in keys]


def comparison_specs(comparison_name):
    config = WCCI_COMPARISON_SPECS[comparison_name]
    return specs_for_keys(config["run_keys"])


def resolved_comparison_specs(comparison_name):
    return wm.resolve_named_plot_specs(
        comparison_specs(comparison_name),
        history=source_history(comparison_name),
    )


def comparison_run_groups(comparison_name):
    return {
        spec["label"]: list(spec.get("runs", []))
        for spec in resolved_comparison_specs(comparison_name)
        if spec.get("runs")
    }


def plot_wcci_comparison(comparison_name, **overrides):
    config = dict(WCCI_COMPARISON_SPECS[comparison_name])
    config.update(overrides)
    run_groups = comparison_run_groups(comparison_name)
    if not run_groups:
        raise RuntimeError(f"No runs resolved for {comparison_name}")
    fig, long_df, mean_df = plot_survival_groups(
        run_groups,
        title=config.get("title", comparison_name),
        split=config.get("split", "test"),
        smooth=config.get("smooth", 5),
        y_range=config.get("y_range", (0, 105)),
        show_members=config.get("show_members", True),
        show_std=config.get("show_std", True),
        save_name=config.get("save_name"),
        history=source_history(comparison_name),
    )
    return {
        "fig": fig,
        "run_groups": run_groups,
        "long": long_df,
        "mean": mean_df,
        "resolved_specs": resolved_comparison_specs(comparison_name),
    }


def plot_wcci_comparison_grid(comparison_names, *, title, save_name, ncols=2):
    # Convenience helper if you want a compact grid later. It resolves the same
    # editable specs, but uses WCCI's wide-history survival frames.
    from plotly.subplots import make_subplots
    names = list(comparison_names)
    ncols = max(1, int(ncols))
    nrows = int(np.ceil(len(names) / ncols))
    subplot_titles = [WCCI_COMPARISON_SPECS[name].get("subplot_title", name) for name in names]
    fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=subplot_titles)
    for idx, name in enumerate(names, start=1):
        row = int(np.ceil(idx / ncols))
        col = ((idx - 1) % ncols) + 1
        run_groups = comparison_run_groups(name)
        long_df = survival_long(run_groups, split="test", smooth=5, history=source_history(name))
        mean_df = mean_survival_curves(long_df)
        for label in run_groups:
            curve_data = mean_df[mean_df["curve"].eq(label)].sort_values("step_millions")
            if curve_data.empty:
                continue
            fig.add_trace(
                go.Scatter(
                    x=curve_data["step_millions"],
                    y=curve_data["mean_survival"],
                    mode="lines",
                    name=label,
                    legendgroup=label,
                    line={"color": WCCI_COLOR.get(label), "width": 3.0},
                    showlegend=idx == 1,
                ),
                row=row,
                col=col,
            )
        fig.update_yaxes(range=[0, 105], title_text="survival (%)", row=row, col=col)
        fig.update_xaxes(title_text="steps (M)", row=row, col=col)
    fig.update_layout(
        title=title,
        template="plotly_white",
        width=1550,
        height=max(520, 420 * nrows),
        hovermode="x unified",
        legend={"x": 1.01, "y": 1, "xanchor": "left", "yanchor": "top"},
        margin={"l": 70, "r": 280, "t": 90, "b": 60},
    )
    save_figure(fig, save_name)
    if SHOW_FIGURES:
        fig.show()
    return {"fig": fig}


def report_comparison_coverage():
    rows = []
    for comparison_name, config in WCCI_COMPARISON_SPECS.items():
        history = source_history(comparison_name)
        available = set(history["run_name"].dropna().astype(str).unique())
        for key in config["run_keys"]:
            spec = WCCI_RUN_SPECS[key]
            resolved = wm.resolve_named_plot_specs([spec], history=history)
            runs = resolved[0].get("runs", []) if resolved else []
            rows.append({
                "comparison": comparison_name,
                "spec_key": key,
                "label": spec.get("label", key),
                "selector": spec.get("prefix") or spec.get("name") or spec.get("regex") or spec.get("contains") or spec.get("runs"),
                "n_runs": len(runs),
                "seeds": sorted(pd.Series([seed_from_run(run) for run in runs]).dropna().astype(int).unique().tolist()),
                "missing": len(runs) == 0,
            })
    coverage = pd.DataFrame(rows)
    display(coverage)
    return coverage


WCCI_COMPARISON_COVERAGE = report_comparison_coverage()



WCCI baselines


,curve,expected,available,missing
0,MLP baseline 20M,3,0,"[wcci_mlp_baseline_72x576_20M_s0, wcci_mlp_bas..."
1,MLP baseline 60M,3,0,"[wcci_mlp_baseline_72x576_60M_s0, wcci_mlp_bas..."
2,reduced MLP baseline lr20m f0.10,3,0,[wcci_reduced_mlp_baseline_72x576_lr20m_f010_s...



WCCI HVG


,curve,expected,available,missing
0,reduced MLP baseline lr20m f0.10,3,0,[wcci_reduced_mlp_baseline_72x576_lr20m_f010_s...
1,global rho heuristic 0.90,3,0,"[wcci_hvg_01_eval_rho090_72x576_s0, wcci_hvg_0..."
2,local rho heuristic 0.90,3,0,"[wcci_hvg_04_eval_local_rho090_72x576_s0, wcci..."
3,gate final-action MAP,3,0,"[wcci_hvg_02_gate_final_map_72x576_s0, wcci_hv..."
4,gate hierarchical greedy,3,0,"[wcci_hvg_03_gate_hierarchical_72x576_s0, wcci..."



WCCI Sparse16


,curve,expected,available,missing
0,reduced MLP baseline lr20m f0.10,3,0,[wcci_reduced_mlp_baseline_72x576_lr20m_f010_s...
1,Sparse16 flat p0.003,3,0,"[wcci_sparse16_flat_p003_72x576_s0, wcci_spars..."



WCCI AIB 24h


,curve,expected,available,missing
0,reduced MLP baseline 24h,3,0,"[wcci_reduced_mlp_baseline_s0, wcci_reduced_ml..."
1,AIB flat local t0.20,3,0,"[wcci_aib_00_flat_local_t020_72x576_s0, wcci_a..."
2,AIB flat local t0.10,3,0,"[wcci_aib_01_flat_local_t010_72x576_s0, wcci_a..."
3,AIB flat local t0.35,3,0,"[wcci_aib_02_flat_local_t035_72x576_s0, wcci_a..."



WCCI AIB


,curve,expected,available,missing
0,reduced MLP baseline lr20m f0.10,3,0,[wcci_reduced_mlp_baseline_72x576_lr20m_f010_s...
1,AIB flat local t0.20,3,0,"[wcci_aib_00_flat_local_t020_72x576_s0, wcci_a..."
2,AIB flat local t0.20 lr20m f0.10,3,0,[wcci_aib_00_flat_local_t020_72x576_lr20m_f010...
3,AIB flat local t0.10,3,0,"[wcci_aib_01_flat_local_t010_72x576_s0, wcci_a..."
4,AIB t0.10 topo0.003,3,0,[wcci_aib_01_flat_local_t010_topo003_72x576_s0...
5,AIB t0.10 topo0.010,3,0,[wcci_aib_01_flat_local_t010_topo010_72x576_s0...


## WCCI Baseline Comparison

Compare the three baseline variants downloaded in `wcci_baseline_2`.


In [ ]:

wcci_baseline_result = plot_wcci_comparison("WCCI baselines")
wcci_baseline_result["fig"]


## WCCI Baseline vs Heuristic / Gate

Compare the selected baseline against global/local heuristic overrides and gated policies from `wcci_hvg_2`.


In [7]:

wcci_hvg_result = plot_wcci_comparison("WCCI HVG")
wcci_hvg_result["fig"]


No survival data for WCCI heuristic vs gate: test episodic survival


## WCCI Baseline vs Sparse16

Compare the selected baseline against the sparse intervention penalty run from `wcci_sparse16_2`.


In [ ]:

wcci_sparse16_result = plot_wcci_comparison("WCCI Sparse16")
wcci_sparse16_result["fig"]


## WCCI Baseline vs AIB

Compare the selected baseline against adaptive intervention budget runs from `wcci_aib_2`.


In [ ]:

wcci_aib_result = plot_wcci_comparison("WCCI AIB")
wcci_aib_result["fig"]


## WCCI Baseline vs All Control Variants

One combined view with the selected baseline and every non-baseline run from the `_2` WCCI groups.


In [ ]:

wcci_all_controls_result = plot_wcci_comparison("WCCI all controls")
wcci_all_controls_result["fig"]


## Final Survival Summary

Mean of the last logged test survival values for every comparison above.


In [ ]:

summary_tables = []
for comparison_name in WCCI_COMPARISON_SPECS:
    run_groups = comparison_run_groups(comparison_name)
    summary = latest_survival_summary(
        run_groups,
        split="test",
        last_n=5,
        history=source_history(comparison_name),
    )
    summary.insert(0, "comparison", comparison_name)
    summary_tables.append(summary)
final_survival_summary = pd.concat(summary_tables, ignore_index=True) if summary_tables else pd.DataFrame()
display(final_survival_summary)
